importing necessary libraries and the dataset

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob
import plotly.graph_objects as go

In [ ]:
folder_path = '/Users/abhimanyuchettiar/Desktop/blinkitdataset/'

In [ ]:

# Load the core files
df_orders = pd.read_csv(folder_path + 'blinkit_orders.csv')
df_items = pd.read_csv(folder_path + 'blinkit_order_items.csv')
df_products = pd.read_csv(folder_path + 'blinkit_products.csv')
df_delivery = pd.read_csv(folder_path + 'blinkit_delivery_performance.csv')
df_feedback = pd.read_csv(folder_path + 'blinkit_customer_feedback.csv')
# Load the marketing file specifically
df_marketing = pd.read_csv(folder_path + 'blinkit_marketing_performance.csv')

# --- STEP 1: MERGING & CLEANING ---

# Merge Items with Products to get Category/Price info
df_master = pd.merge(df_items, df_products, on='product_id', how='left')

# Merge with Orders to get Timestamps and Customer IDs
df_master = pd.merge(df_master, df_orders, on='order_id', how='left')

# Convert dates to datetime objects
df_master['order_date'] = pd.to_datetime(df_master['order_date'])

# Handle missing values in price or ratings
df_master['price'] = df_master['price'].fillna(df_master['price'].median())
df_master['total_price'] = df_master['quantity'] * df_master['price']

In [ ]:
#df_master.colummns
df_master.size

Hourly Sales Peak ( When do people order the most?)

In [ ]:
df_master['hour'] = df_master['order_date'].dt.hour
hourly_sales = df_master.groupby('hour')['total_price'].sum().reset_index()

fig1 = px.line(hourly_sales, x='hour', y='total_price', 
              title='Peak Sales Hours (Quick Commerce Demand)',
              labels={'total_price': 'Total Revenue', 'hour': 'Hour of Day'},
              markers=True, template='plotly_dark')
fig1.show()

Product Category Dominance
which category brings in the most money?

In [ ]:
cat_sales = df_master.groupby('category')['total_price'].sum().reset_index().sort_values('total_price', ascending=False)

fig2 = px.bar(cat_sales, x='category', y='total_price', color='total_price',
             title='Revenue Contribution by Product Category',
             color_continuous_scale='Sunset')
fig2.show()

Delivery Performance (Actual vs Promised Time)

In [ ]:
df_delivery.head()

In [ ]:
# Assuming columns 'actual_time' and 'promised_time
# 1. Convert the columns to datetime objects
df_delivery['actual_time'] = pd.to_datetime(df_delivery['actual_time'])
df_delivery['promised_time'] = pd.to_datetime(df_delivery['promised_time'])

# 2. Now you can subtract them
# This creates a 'Timedelta' object
df_delivery['delay_delta'] = df_delivery['actual_time'] - df_delivery['promised_time']

# 3. Convert that delta into a clean number (e.g., total minutes) for Plotly
df_delivery['delay_minutes'] = df_delivery['delay_delta'].dt.total_seconds() / 60

df_delivery['delay'] = df_delivery['actual_time'] - df_delivery['promised_time']

fig3 = px.histogram(df_delivery, x='delay_minutes', nbins=30,
                   title='Delivery Delay Distribution (Minutes)',
                   color_discrete_sequence=['indianred'],
                   labels={'delay_minutes': 'Delay (Minutes)'})
fig3.add_vline(x=0, line_dash="dash", line_color="green", annotation_text="On Time")
fig3.show()

Customer Feedback Sentiment

In [ ]:
rating_counts = df_feedback['rating'].value_counts().reset_index()
rating_counts.columns = ['rating', 'count']

fig4 = px.pie(rating_counts, values='count', names='rating', 
             title='Customer Satisfaction (Rating Distribution)',
             hole=0.5, color_discrete_sequence=px.colors.sequential.RdBu)
fig4.show()

Inventory Turnover (High Demand vs Stock)
finding which products are fast moving consumer goods

In [ ]:
# Merging inventory with sales count
inventory_check = df_master.groupby('product_name')['quantity'].sum().reset_index()
# Link with stock_level from inventory.csv
# inventory_check = pd.merge(inventory_check, df_inventory, on='product_id')

fig5 = px.scatter(inventory_check, x='quantity', y='product_name', 
                 size='quantity', color='quantity',
                 title='Top Selling Products (Inventory Velocity)')
fig5.show()

Marketing ROI (Sales vs Marketing Spend)

In [ ]:
df_marketing.head()

In [ ]:


# Basic Cleaning using the CORRECT column names from error message
df_marketing['spend'] = df_marketing['spend'].fillna(0)
df_marketing['revenue_generated'] = df_marketing['revenue_generated'].fillna(0)

# 3. Visualization
# Note: Since 'roas' is already a column, we dont need to calculate it
fig6 = px.scatter(df_marketing, 
                 x='spend', 
                 y='revenue_generated', 
                 size='clicks', 
                 color='roas',
                 opacity=0.4,  # This helps see where dots are densest
                 trendline="ols", # Adds a linear regression line
                 hover_name='campaign_name',
                 title='Marketing Campaign Effectiveness (with Trendline)',
                 template='plotly_white')

fig6.show()

so this means there is no correlation between money spent on ads vs the revenue generated by them!

machine learning model implementation 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
X = df_marketing[['spend']] 
y = df_marketing['revenue_generated'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
fig1 = px.scatter(x=X_test['spend'], y=y_test, labels={'x': 'Spend', 'y': 'Actual Revenue'},
                 title=f"Linear Regression: Spend vs Revenue (R²: {r2_score(y_test, y_pred):.4f})")
fig1.add_traces(go.Scatter(x=X_test['spend'], y=y_pred, name='Regression Line', line=dict(color='red')))
fig1.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
features = df_marketing[['spend', 'revenue_generated', 'clicks']]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_marketing['Cluster'] = kmeans.fit_predict(scaled_features)
fig2 = px.scatter(df_marketing, x='spend', y='revenue_generated', color='Cluster',
                 size='clicks', hover_name='campaign_name',
                 title="K-Means Clustering: Campaign Segments",
                 color_continuous_scale='Viridis')
fig2.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import plotly.figure_factory as ff
median_roas = df_marketing['roas'].median()
df_marketing['Success'] = (df_marketing['roas'] > median_roas).astype(int)
X_rf = df_marketing[['spend', 'clicks', 'impressions']]
y_rf = df_marketing['Success']

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)
cm = confusion_matrix(y_test_rf, y_pred_rf)

z = cm
x_labels = ['Predicted Low', 'Predicted High']
y_labels = ['Actual Low', 'Actual High']

fig3 = ff.create_annotated_heatmap(z, x=x_labels, y=y_labels, colorscale='Blues')
fig3.update_layout(title="Random Forest: Confusion Matrix for Campaign Success")
fig3.show()